# Random Projections for Linear Programming — Toy Demo - GUROBI
This notebook is a compact numerical demonstration of the workflow used in the project:

**Original LP → Random Projection → Projected LP → Solution Retrieval → APM**

We consider the standard-form LP

$$\min_x c^T x \quad \text{s.t.}\quad Ax=b,\;x\ge 0.$$

The random projection reduces the number of equality constraints from $m$ to $k<m$. The projected solution is generally not feasible for the original constraints, so we compare a single equality projection with Alternating Projection Method (APM) retrieval. The experiment is illustrative rather than a reproduction of the paper's full benchmark suite.


## Overview


In [21]:
"""
Numerical example for the paper:
'Random Projections for Linear Programming: An Improved Retrieval Phase'

What this script does
---------------------
1. Generate synthetic feasible LP instances in standard form:
       min c^T x
       s.t. A x = b
            x >= 0
2. Solve the original LP.
3. Build a random projection matrix T and solve the projected LP:
       min c^T x
       s.t. T A x = T b
            x >= 0
4. Apply two retrieval methods:
   (a) Single projection (Eq. (5) in the paper)
   (b) APM = alternating projection method
5. Compare:
   - runtime
   - equality feasibility ||A x - b||
   - nonnegativity violation
   - objective gap
6. Plot four figures and save CSV files.


Important note
--------------
- This is a presentation-friendly synthetic experiment, not a full
  reproduction of all paper benchmarks.
- The script is designed to be readable and easy to modify.
"""



"\nNumerical example for the paper:\n'Random Projections for Linear Programming: An Improved Retrieval Phase'\n\nWhat this script does\n---------------------\n1. Generate synthetic feasible LP instances in standard form:\n       min c^T x\n       s.t. A x = b\n            x >= 0\n2. Solve the original LP.\n3. Build a random projection matrix T and solve the projected LP:\n       min c^T x\n       s.t. T A x = T b\n            x >= 0\n4. Apply two retrieval methods:\n   (a) Single projection (Eq. (5) in the paper)\n   (b) APM = alternating projection method\n5. Compare:\n   - runtime\n   - equality feasibility ||A x - b||\n   - nonnegativity violation\n   - objective gap\n6. Plot four figures and save CSV files.\n\n\nImportant note\n--------------\n- This is a presentation-friendly synthetic experiment, not a full\n  reproduction of all paper benchmarks.\n- The script is designed to be readable and easy to modify.\n"

## 1. Import required tools


In [22]:
# =========================
# 1. Import required tools
# =========================

import time  # Used to measure runtime of each stage.
from pathlib import Path  # Used to create output folders and file paths.

import matplotlib.pyplot as plt  # Used to create plots.
import numpy as np  # Used for arrays, linear algebra, and random generation.
import pandas as pd  # Used to store results in DataFrames / CSV files.
from scipy.optimize import linprog  # Used to solve LPs.
from scipy.sparse.linalg import lsmr  # Used for approximate retrieval updates.




## 2. Global configuration for the whole script


In [23]:
# ============================================
# 2. Global configuration for the whole script
# ============================================

# This random seed makes the whole experiment reproducible.
GLOBAL_SEED = 42

# This folder is where all figures and CSV files will be saved.
OUTPUT_DIR = Path.cwd() / "rp_lp_demo_outputs"

# This creates the output folder if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)




## 3. Helper function: random projection T


In [24]:
# ==========================================
# 3. Helper function: random projection T
# ==========================================


def achlioptas_projection_matrix(k: int, m: int, rng: np.random.Generator) -> np.ndarray:
    """
    Create a sparse-ish Achlioptas-style random projection matrix T.

    Parameters
    ----------
    k : int
        Projected dimension (number of rows of T).
    m : int
        Original constraint dimension (number of rows of A).
    rng : np.random.Generator
        Random number generator for reproducibility.

    Returns
    -------
    T : np.ndarray of shape (k, m)
        Random projection matrix.

    Why this form?
    --------------
    This is a common random projection construction:
    entries are in {+1, 0, -1} and then scaled by 1/sqrt(k).

    Compared with dense Gaussian T, this is lighter to compute and is
    still consistent with the general random-projection spirit.
    """
    # Sample each entry independently from {+1, 0, -1}.
    raw = rng.choice(
        [1.0, 0.0, -1.0],
        size=(k, m),
        p=[1.0 / 6.0, 2.0 / 3.0, 1.0 / 6.0],
    )

    # Scale by 1/sqrt(k), which is standard for random projection matrices.
    T = raw / np.sqrt(k)

    # Return the projection matrix.
    return T




## Setup

In [25]:
# =========================
# 1. Import required tools
# =========================

import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import gurobipy as gp
from gurobipy import GRB

from scipy.sparse.linalg import lsmr

## 2. Global configuration for the whole script


In [26]:
# ============================================
# 2. Global configuration for the whole script
# ============================================

# This random seed makes the whole experiment reproducible.
GLOBAL_SEED = 42

# Project root folder
PROJECT_ROOT = Path.cwd().parent.parent

# Save experiment outputs outside the notebook/code folder
OUTPUT_DIR = PROJECT_ROOT / "results" / "gurobi_toy_example"

# Create the folder if it does not already exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)




## 3. Helper function: random projection T


In [27]:
# ==========================================
# 3. Helper function: random projection T
# ==========================================


def achlioptas_projection_matrix(k: int, m: int, rng: np.random.Generator) -> np.ndarray:
    """
    Create a sparse-ish Achlioptas-style random projection matrix T.

    Parameters
    ----------
    k : int
        Projected dimension (number of rows of T).
    m : int
        Original constraint dimension (number of rows of A).
    rng : np.random.Generator
        Random number generator for reproducibility.

    Returns
    -------
    T : np.ndarray of shape (k, m)
        Random projection matrix.

    Why this form?
    --------------
    This is a common random projection construction:
    entries are in {+1, 0, -1} and then scaled by 1/sqrt(k).

    Compared with dense Gaussian T, this is lighter to compute and is
    still consistent with the general random-projection spirit.
    """
    # Sample each entry independently from {+1, 0, -1}.
    raw = rng.choice(
        [1.0, 0.0, -1.0],
        size=(k, m),
        p=[1.0 / 6.0, 2.0 / 3.0, 1.0 / 6.0],
    )

    # Scale by 1/sqrt(k), which is standard for random projection matrices.
    T = raw / np.sqrt(k)

    # Return the projection matrix.
    return T




## 4. Helper function: build a feasible LP in standard form


In [28]:
# =======================================================
# 4. Helper function: build a feasible LP in standard form
# =======================================================


def generate_feasible_lp(m: int, n: int, seed: int):
    """
    Generate a synthetic feasible LP:
        min c^T x
        s.t. A x = b
             x >= 0

    Construction idea
    -----------------
    1. Randomly generate a nonnegative vector x_true.
    2. Randomly generate a matrix A.
    3. Define b = A x_true.

    Then x_true is automatically feasible.

    Parameters
    ----------
    m : int
        Number of equality constraints.
    n : int
        Number of decision variables.
    seed : int
        Seed for the local RNG.

    Returns
    -------
    A : np.ndarray, shape (m, n)
        Constraint matrix.
    b : np.ndarray, shape (m,)
        Right-hand side vector.
    c : np.ndarray, shape (n,)
        Objective vector.
    x_true : np.ndarray, shape (n,)
        A known feasible point used to build b.
    upper_bound : float
        Common upper bound for all variables in the LP solver.
    """
    # Create a local RNG so each problem instance is reproducible.
    rng = np.random.default_rng(seed)

    # Randomly generate a dense matrix A.
    A = rng.standard_normal((m, n))

    # Create a sparse-ish nonnegative feasible vector x_true.
    x_true = np.zeros(n)

    # Select exactly m active coordinates to make x_true easier to interpret.
    active_idx = rng.choice(n, size=m, replace=False)

    # Assign positive values on the chosen active coordinates.
    x_true[active_idx] = rng.uniform(0.2, 2.0, size=m)

    # Define b = A x_true so that x_true is feasible by construction.
    b = A @ x_true

    # Create a nonnegative objective vector c.
    c = rng.uniform(0.0, 1.0, size=n)

    # Set a simple common upper bound for the LP variables.
    upper_bound = 5.0

    # Return everything needed for the LP.
    return A, b, c, x_true, upper_bound




## 5. Helper function: compute performance metrics


In [29]:
# ============================================
# 5. Helper function: compute performance metrics
# ============================================


def compute_metrics(A: np.ndarray, b: np.ndarray, c: np.ndarray, x: np.ndarray) -> dict:
    """
    Compute the key metrics we want to compare across methods.

    Metrics
    -------
    objective
        c^T x

    eq_residual
        ||A x - b||_2
        This measures how badly the equality constraints are violated.

    negativity_sum
        sum_i max(-x_i, 0)
        This measures total nonnegativity violation.

    min_entry
        min_i x_i
        This tells us whether there are negative entries.

    Parameters
    ----------
    A, b, c, x : np.ndarray
        Standard LP ingredients.

    Returns
    -------
    metrics : dict
        Dictionary of scalar metrics.
    """
    # Compute the equality-constraint residual vector.
    residual_vector = A @ x - b

    # Compute the total amount of negative mass in x.
    negative_part_sum = np.sum(np.maximum(-x, 0.0))

    # Build the metric dictionary.
    metrics = {
        "objective": float(c @ x),
        "eq_residual": float(np.linalg.norm(residual_vector, ord=2)),
        "negativity_sum": float(negative_part_sum),
        "min_entry": float(np.min(x)),
    }

    # Return the metrics.
    return metrics




## 6. Solve the original LP (baseline method)

In [30]:
# ==========================================
# 6. Solve the original LP (baseline method)
# ==========================================

def solve_original_lp(
    A: np.ndarray,
    b: np.ndarray,
    c: np.ndarray,
    upper_bound: float
):
    """
    Solve the original LP:

        minimize    c^T x

        subject to  A x = b
                    0 <= x <= upper_bound

    Parameters
    ----------
    A : np.ndarray
        Constraint matrix with shape (m, n).

    b : np.ndarray
        Right-hand-side vector with length m.

    c : np.ndarray
        Objective coefficient vector with length n.

    upper_bound : float
        Maximum allowed value for each decision variable.

    Returns
    -------
    x_opt : np.ndarray
        Optimal decision-variable values.

    runtime : float
        Wall-clock time spent solving the LP.

    model : gurobipy.Model
        Gurobi model object.
    """

    # A has shape (m, n), where:
    # m = number of constraints
    # n = number of decision variables
    n = A.shape[1]

    # Create a new Gurobi optimization model.
    model = gp.Model("original_lp")

    # Turn off Gurobi's console output so repeated experiments stay clean.
    model.Params.OutputFlag = 0

    # Create the decision-variable vector x.
    #
    # shape=n:
    #     creates n decision variables.
    #
    # lb=0.0:
    #     enforces x >= 0.
    #
    # ub=upper_bound:
    #     enforces x <= upper_bound.
    #
    # vtype=GRB.CONTINUOUS:
    #     tells Gurobi that the variables are continuous,
    #     which makes this a linear program rather than an integer program.
    x = model.addMVar(
        shape=n,
        lb=0.0,
        ub=upper_bound,
        vtype=GRB.CONTINUOUS,
        name="x"
    )

    # Set the objective function:
    #
    # minimize c^T x
    #
    # The @ operator performs matrix/vector multiplication.
    model.setObjective(
        c @ x,
        GRB.MINIMIZE
    )

    # Add the equality constraints:
    #
    # A x = b
    #
    # Gurobi's matrix interface allows the mathematical
    # expression to be written almost directly in Python.
    model.addConstr(
        A @ x == b,
        name="equality_constraints"
    )

    # Start the wall-clock timer immediately before solving.
    start_time = time.perf_counter()

    # Solve the linear program using Gurobi.
    model.optimize()

    # Calculate elapsed wall-clock solve time.
    runtime = time.perf_counter() - start_time

    # Check that Gurobi actually found an optimal solution.
    #
    # GRB.OPTIMAL corresponds to Gurobi status code 2.
    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(
            f"Original LP failed. Gurobi status code: {model.Status}"
        )

    # x is currently a Gurobi variable object.
    #
    # x.X retrieves the numerical values Gurobi found.
    # .copy() converts/copies them into a regular NumPy array.
    x_opt = x.X.copy()

    # Return:
    # 1. the optimal solution
    # 2. the measured wall-clock runtime
    # 3. the Gurobi model object
    return x_opt, runtime, model

## 7. Solve the projected LP after random projection


In [31]:
# ============================================
# 7. Solve the projected LP after random projection
# ============================================

def solve_projected_lp(
    A: np.ndarray,
    b: np.ndarray,
    c: np.ndarray,
    k: int,
    upper_bound: float,
    seed: int,
):
    """
    Solve the randomly projected LP:

        minimize    c^T x

        subject to  T A x = T b
                    0 <= x <= upper_bound

    Parameters
    ----------
    A : np.ndarray
        Original constraint matrix with shape (m, n).

    b : np.ndarray
        Original right-hand-side vector with length m.

    c : np.ndarray
        Objective coefficient vector with length n.

    k : int
        Number of constraints after projection.
        Usually k < m.

    upper_bound : float
        Maximum value allowed for each decision variable.

    seed : int
        Random seed used to generate the projection matrix T.

    Returns
    -------
    x_proj : np.ndarray
        Optimal solution of the projected LP.

    T : np.ndarray
        Random projection matrix used in this run.

    runtime : float
        Wall-clock time spent solving the projected LP.

    model : gurobipy.Model
        Gurobi model object.
    """

    # Create a local random-number generator.
    #
    # Using a seed makes the random projection reproducible.
    # Running the function again with the same seed gives the same T.
    rng = np.random.default_rng(seed)

    # A has shape (m, n).
    # The number of rows is the original number of equality constraints m.
    m = A.shape[0]

    # Generate the k x m random projection matrix T.
    #
    # T compresses the original m equality constraints down to k constraints.
    T = achlioptas_projection_matrix(
        k=k,
        m=m,
        rng=rng
    )

    # Apply the random projection to both sides of Ax = b.
    #
    # Original:
    #     A x = b
    #
    # Projected:
    #     T A x = T b
    A_projected = T @ A
    b_projected = T @ b

    # A_projected now has shape (k, n),
    # instead of the original shape (m, n).

    # Get the number of decision variables n.
    n = A.shape[1]

    # Create a new Gurobi model for the projected LP.
    model = gp.Model("projected_lp")

    # Suppress Gurobi console output during experiments.
    model.Params.OutputFlag = 0

    # Create the same n continuous decision variables as the original LP.
    #
    # Important:
    # We are reducing the number of CONSTRAINTS,
    # not the number of decision variables.
    x = model.addMVar(
        shape=n,
        lb=0.0,
        ub=upper_bound,
        vtype=GRB.CONTINUOUS,
        name="x"
    )

    # Keep the exact same objective function as the original LP:
    #
    # minimize c^T x
    model.setObjective(
        c @ x,
        GRB.MINIMIZE
    )

    # Add the PROJECTED equality constraints:
    #
    # T A x = T b
    #
    # This is the key difference from Section 6.
    model.addConstr(
        A_projected @ x == b_projected,
        name="projected_constraints"
    )

    # Start the wall-clock timer immediately before solving.
    start_time = time.perf_counter()

    # Solve the projected LP.
    model.optimize()

    # Measure wall-clock solve time.
    runtime = time.perf_counter() - start_time

    # Verify that Gurobi found an optimal solution.
    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(
            f"Projected LP failed. Gurobi status code: {model.Status}"
        )

    # Extract the numerical projected solution.
    x_proj = x.X.copy()

    # Return the projected solution, projection matrix,
    # runtime, and Gurobi model.
    return x_proj, T, runtime, model

## 8. Retrieval helper: exact equality projection update


In [32]:
# =====================================================
# 8. Retrieval helper: exact equality projection update
# =====================================================

def exact_equality_projection_update(
    A: np.ndarray,
    b: np.ndarray,
    x: np.ndarray
) -> np.ndarray:
    """
    Project the current point x onto the equality constraint set:

        A x = b

    The projection formula is:

        x_new = x + A^T (A A^T)^(-1) (b - A x)

    Instead of explicitly computing the inverse of A A^T,
    we solve a linear system, which is more numerically stable.

    Parameters
    ----------
    A : np.ndarray
        Original constraint matrix.

    b : np.ndarray
        Original right-hand-side vector.

    x : np.ndarray
        Current point being projected.

    Returns
    -------
    x_new : np.ndarray
        Point projected onto the equality constraint set A x = b.
    """

    # Compute how far the current point is from satisfying A x = b.
    #
    # If x already satisfies the equality constraints,
    # then this residual will be approximately zero.
    residual = b - A @ x

    # Form the matrix A A^T.
    #
    # If A has shape (m, n), then:
    #
    # A       -> (m, n)
    # A.T     -> (n, m)
    # A @ A.T -> (m, m)
    gram = A @ A.T

    # Solve:
    #
    #     (A A^T) y = b - A x
    #
    # We solve for y instead of explicitly calculating
    # (A A^T)^(-1).
    y = np.linalg.solve(
        gram,
        residual
    )

    # Convert the correction y back into x-space using A^T.
    #
    # This implements:
    #
    # x_new = x + A^T y
    #
    # which is equivalent to:
    #
    # x_new = x + A^T (A A^T)^(-1)(b - A x)
    x_new = x + A.T @ y

    # Return the equality-feasible point.
    return x_new

## 9. Retrieval helper: approximate equality projection update


In [33]:
# ==========================================================
# 9. Retrieval helper: approximate equality projection update
# ==========================================================


def approximate_equality_projection_update(A: np.ndarray, b: np.ndarray, x: np.ndarray) -> np.ndarray:
    """
    Approximate the equality projection step using LSMR.

    Why include this?
    -----------------
    Section 3.5 of the paper discusses replacing the expensive pseudoinverse
    calculation by iterative methods such as LSMR / LSQR.

    Here we use LSMR to solve:
        (A A^T) y = b - A x
    approximately.

    Parameters
    ----------
    A, b, x : np.ndarray
        Standard LP ingredients and current iterate.

    Returns
    -------
    x_new : np.ndarray
        Approximate equality-projected point.
    """
    # Compute the equality residual.
    residual = b - A @ x

    # Build the matrix A A^T.
    gram = A @ A.T

    # Use LSMR to approximately solve gram * y = residual.
    # The first entry in LSMR output is the approximate solution vector y.
    y = lsmr(gram, residual, atol=1e-8, btol=1e-8, maxiter=500)[0]

    # Apply the corresponding correction in x-space.
    x_new = x + A.T @ y

    # Return the approximate update.
    return x_new




## 10. Single projection retrieval (one-shot)


In [34]:
# =============================================
# 10. Single projection retrieval (one-shot)
# =============================================


def single_projection_retrieval(
    A: np.ndarray,
    b: np.ndarray,
    x_proj: np.ndarray,
    mode: str = "exact",
):
    """
    Apply a single equality projection step to the projected LP solution.

    This corresponds to a one-shot retrieval:
        projected solution -> Eq. (5) once

    Parameters
    ----------
    A, b : np.ndarray
        Original LP data.

    x_proj : np.ndarray
        Solution of the projected LP.

    mode : str
        "exact"       -> use numpy.linalg.solve
        "approx_lsmr" -> use scipy.sparse.linalg.lsmr

    Returns
    -------
    x_single : np.ndarray
        Retrieved solution after one equality projection.

    runtime : float
        Runtime of this one-shot retrieval step.
    """

    # Start timing the retrieval step.
    start_time = time.perf_counter()

    # Choose which equality projection method to use.
    if mode == "exact":

        # Use the exact projection from Section 8.
        x_single = exact_equality_projection_update(
            A,
            b,
            x_proj
        )

    elif mode == "approx_lsmr":

        # Use the approximate LSMR projection from Section 9.
        x_single = approximate_equality_projection_update(
            A,
            b,
            x_proj
        )

    else:

        # Stop if an unsupported retrieval method was requested.
        raise ValueError(
            "mode must be 'exact' or 'approx_lsmr'"
        )

    # Measure total retrieval runtime.
    runtime = time.perf_counter() - start_time

    # Return the retrieved solution and its runtime.
    return x_single, runtime

## 11. Alternating Projection Method (APM) retrieval


In [35]:
# ======================================================
# 11. Alternating Projection Method (APM) retrieval
# ======================================================


def apm_retrieval(
    A: np.ndarray,
    b: np.ndarray,
    x_start: np.ndarray,
    mode: str = "exact",
    max_iter: int = 30,
    delta: float = 1e-6,
):
    """
    Alternating Projection Method (APM).

    Loop structure
    --------------
    Starting from x_start:

    1. Project onto the equality set A x = b.
    2. If x is already nonnegative, stop.
    3. Clip negative entries to zero (project onto x >= 0).
    4. If the equality residual is small enough, stop.
    5. Repeat.

    Parameters
    ----------
    A, b : np.ndarray
        Original LP data.
    x_start : np.ndarray
        Initial point for APM. In our experiments this is x_proj.
    mode : str
        "exact" or "approx_lsmr" for the equality projection step.
    max_iter : int
        Maximum number of APM iterations.
    delta : float
        Tolerance for ||A x - b||_2.

    Returns
    -------
    x_current : np.ndarray
        Final APM output.
    runtime : float
        Total runtime of the whole APM loop.
    history_df : pd.DataFrame
        Per-iteration diagnostics:
        - iteration number
        - equality residual
        - negativity sum
        - minimum entry
    """
    # Copy the starting point so we do not modify the original array by accident.
    x_current = x_start.copy()

    # This list will store diagnostics from each APM iteration.
    history_rows = []

    # Record overall start time.
    start_time = time.perf_counter()

    # Run the APM loop for at most max_iter steps.
    for iteration in range(1, max_iter + 1):
        # ----- Step A: project onto A x = b -----
        if mode == "exact":
            x_current = exact_equality_projection_update(A, b, x_current)
        elif mode == "approx_lsmr":
            x_current = approximate_equality_projection_update(A, b, x_current)
        else:
            raise ValueError("mode must be 'exact' or 'approx_lsmr'")

        # Compute diagnostics right after the equality projection step.
        metrics_after_eq = compute_metrics(A, b, np.zeros_like(x_current), x_current)

        # If x is already nonnegative, we can stop here.
        if np.min(x_current) >= -1e-10:
            history_rows.append(
                {
                    "iteration": iteration,
                    "stage": "after_eq_projection_stop",
                    "eq_residual": metrics_after_eq["eq_residual"],
                    "negativity_sum": metrics_after_eq["negativity_sum"],
                    "min_entry": metrics_after_eq["min_entry"],
                }
            )
            break

        # ----- Step B: project onto x >= 0 by clipping negative entries -----
        x_current = np.maximum(x_current, 0.0)

        # Compute diagnostics after the nonnegativity projection.
        metrics_after_clip = compute_metrics(A, b, np.zeros_like(x_current), x_current)

        # Save the iteration diagnostics.
        history_rows.append(
            {
                "iteration": iteration,
                "stage": "after_clip",
                "eq_residual": metrics_after_clip["eq_residual"],
                "negativity_sum": metrics_after_clip["negativity_sum"],
                "min_entry": metrics_after_clip["min_entry"],
            }
        )

        # If the equality residual is small enough, stop.
        if metrics_after_clip["eq_residual"] <= delta:
            break

    # Measure total APM runtime.
    runtime = time.perf_counter() - start_time

    # Convert the history into a DataFrame for easier plotting / saving.
    history_df = pd.DataFrame(history_rows)

    # Return the final point, runtime, and diagnostics table.
    return x_current, runtime, history_df




## 12. Run the whole experiment over multiple sizes


In [36]:
# ===================================================
# 12. Run the whole experiment over multiple sizes
# ===================================================


def run_experiment(
    sizes,
    n_factor: int = 2,
    projection_ratio: float = 0.4,
    replications: int = 5,
    retrieval_mode: str = "exact",
):
    """
    Run the full numerical experiment for multiple LP sizes.

    Parameters
    ----------
    sizes : list[int]
        Different values of m (number of equality constraints).

    n_factor : int
        Number of variables is n = n_factor * m.

    projection_ratio : float
        Projected size is k = projection_ratio * m.

    replications : int
        Number of random instances per size.

    retrieval_mode : str
        "exact" or "approx_lsmr"

    Returns
    -------
    runs_df : pd.DataFrame
        Raw results for all runs.

    apm_trace_df : pd.DataFrame
        Per-iteration APM traces across all runs.

    summary_df : pd.DataFrame
        Average results aggregated by problem size.
    """

    # Create lists to accumulate:
    # 1. run-level results
    # 2. APM iteration-level results
    run_rows = []
    trace_rows = []

    # Loop through each original LP size.
    #
    # Here, m is the number of equality constraints.
    for m in sizes:

        # Set the number of decision variables.
        #
        # Example:
        # m = 20 and n_factor = 2
        # gives n = 40 variables.
        n = n_factor * m

        # Set the projected dimension k.
        #
        # Example:
        # m = 20 and projection_ratio = 0.4
        # gives k = 8 projected constraints.
        #
        # max(4, ...) prevents k from becoming extremely small.
        k = max(
            4,
            int(round(projection_ratio * m))
        )

        # Repeat each problem size multiple times.
        #
        # This helps account for randomness in both:
        # - the generated LP
        # - the random projection
        for rep in range(replications):

            # Create a reproducible seed for this specific run.
            seed = 1000 + 100 * m + rep

            # -----------------------------------------
            # Generate the original feasible LP
            # -----------------------------------------

            A, b, c, x_true, upper_bound = generate_feasible_lp(
                m=m,
                n=n,
                seed=seed
            )

            # -----------------------------------------
            # Solve the ORIGINAL LP
            # -----------------------------------------
            #
            # This now uses Gurobi because we converted
            # solve_original_lp() in Section 6.
            x_original, time_original, _ = solve_original_lp(
                A,
                b,
                c,
                upper_bound
            )

            # -----------------------------------------
            # Solve the PROJECTED LP
            # -----------------------------------------
            #
            # This also now uses Gurobi because we converted
            # solve_projected_lp() in Section 7.
            x_projected, T, time_projected, _ = solve_projected_lp(
                A,
                b,
                c,
                k,
                upper_bound,
                seed=seed + 999
            )

            # -----------------------------------------
            # Single-projection retrieval
            # -----------------------------------------
            #
            # Apply one equality-projection retrieval step.
            x_single, time_single = single_projection_retrieval(
                A=A,
                b=b,
                x_proj=x_projected,
                mode=retrieval_mode,
            )

            # -----------------------------------------
            # APM retrieval
            # -----------------------------------------
            #
            # Start from the projected solution and run
            # the alternating projection retrieval routine.
            x_apm, time_apm, history_df = apm_retrieval(
                A=A,
                b=b,
                x_start=x_projected,
                mode=retrieval_mode,
                max_iter=30,
                delta=1e-6,
            )

            # -----------------------------------------
            # Compute solution-quality metrics
            # -----------------------------------------

            metrics_original = compute_metrics(
                A,
                b,
                c,
                x_original
            )

            metrics_projected = compute_metrics(
                A,
                b,
                c,
                x_projected
            )

            metrics_single = compute_metrics(
                A,
                b,
                c,
                x_single
            )

            metrics_apm = compute_metrics(
                A,
                b,
                c,
                x_apm
            )

            # -----------------------------------------
            # Compute relative objective gaps
            # -----------------------------------------

            # Use the original LP objective as the baseline.
            #
            # max(1e-8, ...) prevents division by zero if
            # the optimal objective is extremely close to 0.
            obj_base = max(
                1e-8,
                abs(metrics_original["objective"])
            )

            projected_rel_gap = (
                abs(
                    metrics_projected["objective"]
                    - metrics_original["objective"]
                )
                / obj_base
            )

            single_rel_gap = (
                abs(
                    metrics_single["objective"]
                    - metrics_original["objective"]
                )
                / obj_base
            )

            apm_rel_gap = (
                abs(
                    metrics_apm["objective"]
                    - metrics_original["objective"]
                )
                / obj_base
            )

            # -----------------------------------------
            # Save results from this run
            # -----------------------------------------

            run_rows.append(
                {
                    "m": m,
                    "n": n,
                    "k": k,
                    "replication": rep,
                    "retrieval_mode": retrieval_mode,

                    # Runtime results
                    "time_original": time_original,
                    "time_projected": time_projected,
                    "time_single": time_single,
                    "time_apm": time_apm,

                    # Total RP workflow time:
                    # projected LP solve + APM retrieval
                    "time_projected_plus_apm":
                        time_projected + time_apm,

                    # Runtime ratios
                    "ratio_projected_vs_original":
                        time_projected / time_original,

                    "ratio_projected_plus_apm_vs_original":
                        (time_projected + time_apm)
                        / time_original,

                    # Objective values
                    "original_objective":
                        metrics_original["objective"],

                    "projected_objective":
                        metrics_projected["objective"],

                    "single_objective":
                        metrics_single["objective"],

                    "apm_objective":
                        metrics_apm["objective"],

                    # Equality feasibility errors
                    "projected_eq_residual":
                        metrics_projected["eq_residual"],

                    "single_eq_residual":
                        metrics_single["eq_residual"],

                    "apm_eq_residual":
                        metrics_apm["eq_residual"],

                    # Non-negativity violations
                    "projected_negativity":
                        metrics_projected["negativity_sum"],

                    "single_negativity":
                        metrics_single["negativity_sum"],

                    "apm_negativity":
                        metrics_apm["negativity_sum"],

                    # Relative objective gaps
                    "projected_rel_obj_gap":
                        projected_rel_gap,

                    "single_rel_obj_gap":
                        single_rel_gap,

                    "apm_rel_obj_gap":
                        apm_rel_gap,

                    # Number of APM iterations recorded
                    "apm_iterations_recorded":
                        len(history_df),
                }
            )

            # -----------------------------------------
            # Save APM iteration history
            # -----------------------------------------

            if len(history_df) > 0:

                # Make a copy so we can add identifying
                # information for this experiment.
                history_copy = history_df.copy()

                history_copy["m"] = m
                history_copy["n"] = n
                history_copy["k"] = k
                history_copy["replication"] = rep
                history_copy["retrieval_mode"] = retrieval_mode

                trace_rows.append(history_copy)

    # ==========================================
    # Convert collected results into DataFrames
    # ==========================================

    # One row for every experimental run.
    runs_df = pd.DataFrame(run_rows)

    # Combine all APM iteration histories.
    if len(trace_rows) > 0:
        apm_trace_df = pd.concat(
            trace_rows,
            ignore_index=True
        )
    else:
        apm_trace_df = pd.DataFrame()

    # ==========================================
    # Compute average results for each LP size
    # ==========================================

    summary_df = (
        runs_df.groupby(
            ["m", "retrieval_mode"],
            as_index=False
        )
        .agg(
            time_original=("time_original", "mean"),
            time_projected=("time_projected", "mean"),
            time_single=("time_single", "mean"),
            time_apm=("time_apm", "mean"),

            time_projected_plus_apm=(
                "time_projected_plus_apm",
                "mean"
            ),

            ratio_projected_vs_original=(
                "ratio_projected_vs_original",
                "mean"
            ),

            ratio_projected_plus_apm_vs_original=(
                "ratio_projected_plus_apm_vs_original",
                "mean"
            ),

            projected_eq_residual=(
                "projected_eq_residual",
                "mean"
            ),

            single_eq_residual=(
                "single_eq_residual",
                "mean"
            ),

            apm_eq_residual=(
                "apm_eq_residual",
                "mean"
            ),

            projected_negativity=(
                "projected_negativity",
                "mean"
            ),

            single_negativity=(
                "single_negativity",
                "mean"
            ),

            apm_negativity=(
                "apm_negativity",
                "mean"
            ),

            projected_rel_obj_gap=(
                "projected_rel_obj_gap",
                "mean"
            ),

            single_rel_obj_gap=(
                "single_rel_obj_gap",
                "mean"
            ),

            apm_rel_obj_gap=(
                "apm_rel_obj_gap",
                "mean"
            ),

            apm_iterations_recorded=(
                "apm_iterations_recorded",
                "mean"
            ),
        )
    )

    # Return all three result tables.
    return runs_df, apm_trace_df, summary_df

## 13. Plotting helper for presentation figures


In [37]:
# ===========================================
# 13. Plotting helper for presentation figures
# ===========================================


def make_plots(summary_df: pd.DataFrame, retrieval_mode: str, output_dir: Path):
    """
    Create and save four presentation-friendly figures.

    Parameters
    ----------
    summary_df : pd.DataFrame
        Aggregated summary results.
    retrieval_mode : str
        Either "exact" or "approx_lsmr", used in filenames.
    output_dir : Path
        Folder where the figures will be saved.

    Returns
    -------
    plot_paths : dict
        Dictionary mapping plot names to their saved file paths.
    """
    # Filter summary rows for the chosen retrieval mode.
    data = summary_df[summary_df["retrieval_mode"] == retrieval_mode].copy()

    # Sort by m so the x-axis is ordered nicely.
    data = data.sort_values("m")

    # This dictionary will store the final file paths.
    plot_paths = {}

    # ------------------------------------------
    # Plot 1: Runtime comparison
    # ------------------------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(data["m"], data["time_original"], marker="o", label="Original LP")
    plt.plot(data["m"], data["time_projected"], marker="o", label="Projected LP only")
    plt.plot(data["m"], data["time_projected_plus_apm"], marker="o", label="Projected LP + APM")
    plt.xlabel("Number of original constraints m")
    plt.ylabel("Average runtime (seconds)")
    plt.title(f"Runtime comparison ({retrieval_mode})")
    plt.legend()
    plt.tight_layout()

    runtime_path = output_dir / f"runtime_comparison_{retrieval_mode}.png"
    plt.savefig(runtime_path, dpi=200)
    plt.close()

    plot_paths["runtime"] = runtime_path

    # ------------------------------------------
    # Plot 2: Equality feasibility
    # ------------------------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(data["m"], data["projected_eq_residual"], marker="o", label="Projected solution")
    plt.plot(data["m"], data["single_eq_residual"], marker="o", label="Single projection")
    plt.plot(data["m"], data["apm_eq_residual"], marker="o", label="APM")
    plt.yscale("log")
    plt.xlabel("Number of original constraints m")
    plt.ylabel("Average ||A x - b||")
    plt.title(f"Equality-feasibility improvement ({retrieval_mode})")
    plt.legend()
    plt.tight_layout()

    feasibility_path = output_dir / f"feasibility_improvement_{retrieval_mode}.png"
    plt.savefig(feasibility_path, dpi=200)
    plt.close()

    plot_paths["feasibility"] = feasibility_path

    # ------------------------------------------
    # Plot 3: Nonnegativity violation
    # ------------------------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(data["m"], data["projected_negativity"], marker="o", label="Projected solution")
    plt.plot(data["m"], data["single_negativity"], marker="o", label="Single projection")
    plt.plot(data["m"], data["apm_negativity"], marker="o", label="APM")
    plt.yscale("log")
    plt.xlabel("Number of original constraints m")
    plt.ylabel("Average sum of negative parts")
    plt.title(f"Nonnegativity improvement ({retrieval_mode})")
    plt.legend()
    plt.tight_layout()

    negativity_path = output_dir / f"nonnegativity_improvement_{retrieval_mode}.png"
    plt.savefig(negativity_path, dpi=200)
    plt.close()

    plot_paths["nonnegativity"] = negativity_path

    # ------------------------------------------
    # Plot 4: Relative objective gap
    # ------------------------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(data["m"], data["projected_rel_obj_gap"], marker="o", label="Projected solution")
    plt.plot(data["m"], data["single_rel_obj_gap"], marker="o", label="Single projection")
    plt.plot(data["m"], data["apm_rel_obj_gap"], marker="o", label="APM")
    plt.xlabel("Number of original constraints m")
    plt.ylabel("Average relative objective gap")
    plt.title(f"Objective quality vs original optimum ({retrieval_mode})")
    plt.legend()
    plt.tight_layout()

    obj_gap_path = output_dir / f"objective_gap_{retrieval_mode}.png"
    plt.savefig(obj_gap_path, dpi=200)
    plt.close()

    plot_paths["objective_gap"] = obj_gap_path

    # Return the saved paths.
    return plot_paths




## 14. Main script entry point


In [38]:
# ===========================
# 14. Main script entry point
# ===========================


def main():
    """
    Main function:
    1. Set experiment sizes and parameters.
    2. Run experiments using Gurobi for the original and projected LP solves.
    3. Compare exact and approximate retrieval methods.
    4. Save CSV files and figures.
    5. Print a compact summary to the terminal.
    """

    # Choose several original constraint sizes m.
    sizes = [20, 30, 40, 50, 60]

    # Set n = n_factor * m.
    n_factor = 2

    # Set projected size k = projection_ratio * m.
    projection_ratio = 0.4

    # Number of random instances per size.
    replications = 5

    # Run the experiment with exact retrieval updates.
    runs_exact, trace_exact, summary_exact = run_experiment(
        sizes=sizes,
        n_factor=n_factor,
        projection_ratio=projection_ratio,
        replications=replications,
        retrieval_mode="exact",
    )

    # Run the experiment with approximate LSMR retrieval updates.
    runs_lsmr, trace_lsmr, summary_lsmr = run_experiment(
        sizes=sizes,
        n_factor=n_factor,
        projection_ratio=projection_ratio,
        replications=replications,
        retrieval_mode="approx_lsmr",
    )

    # Combine exact and approximate run-level results.
    runs_df = pd.concat(
        [runs_exact, runs_lsmr],
        ignore_index=True
    )

    # Combine exact and approximate summary tables.
    summary_df = pd.concat(
        [summary_exact, summary_lsmr],
        ignore_index=True
    )

    # Combine the APM iteration traces.
    if len(trace_exact) > 0 and len(trace_lsmr) > 0:
        apm_trace_df = pd.concat(
            [trace_exact, trace_lsmr],
            ignore_index=True
        )

    elif len(trace_exact) > 0:
        apm_trace_df = trace_exact.copy()

    elif len(trace_lsmr) > 0:
        apm_trace_df = trace_lsmr.copy()

    else:
        apm_trace_df = pd.DataFrame()

    # Define output file paths.
    runs_path = OUTPUT_DIR / "all_runs.csv"
    summary_path = OUTPUT_DIR / "summary_by_size.csv"
    trace_path = OUTPUT_DIR / "apm_trace.csv"

    # Save experiment results to CSV files.
    runs_df.to_csv(
        runs_path,
        index=False
    )

    summary_df.to_csv(
        summary_path,
        index=False
    )

    apm_trace_df.to_csv(
        trace_path,
        index=False
    )

    # Make figures for exact retrieval.
    exact_plot_paths = make_plots(
        summary_df,
        retrieval_mode="exact",
        output_dir=OUTPUT_DIR
    )

    # Make figures for approximate LSMR retrieval.
    lsmr_plot_paths = make_plots(
        summary_df,
        retrieval_mode="approx_lsmr",
        output_dir=OUTPUT_DIR
    )

    # Print a compact summary for quick inspection.
    print("\n=== Summary table ===")
    print(
        summary_df
        .round(6)
        .to_string(index=False)
    )

    print("\n=== Saved CSV files ===")
    print(runs_path)
    print(summary_path)
    print(trace_path)

    print("\n=== Saved exact-mode plots ===")
    for name, path in exact_plot_paths.items():
        print(f"{name}: {path}")

    print("\n=== Saved approx_lsmr-mode plots ===")
    for name, path in lsmr_plot_paths.items():
        print(f"{name}: {path}")


# Run main() when this file/notebook section is executed directly.
if __name__ == "__main__":
    main()


=== Summary table ===
 m retrieval_mode  time_original  time_projected  time_single  time_apm  time_projected_plus_apm  ratio_projected_vs_original  ratio_projected_plus_apm_vs_original  projected_eq_residual  single_eq_residual  apm_eq_residual  projected_negativity  single_negativity  apm_negativity  projected_rel_obj_gap  single_rel_obj_gap  apm_rel_obj_gap  apm_iterations_recorded
20          exact       0.000745        0.000346     0.000073  0.001646                 0.001992                     0.465419                              2.690512              20.978801            0.000000         0.080505                   0.0           4.992198             0.0               0.648061            0.453265         0.209949                     30.0
30          exact       0.001407        0.000516     0.000092  0.002475                 0.002991                     0.371467                              2.182635              33.140327            0.000000         0.024213                   0.0